### TOOLS

Models can request to call tools that performs tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A Function or coroutine to execute.

In [3]:

import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Hmm, I know that parrots are known for mimicking human speech, but why exactly do they do that? Let me start by recalling what I know about birds and their communication. Most birds use songs for various reasons like attracting mates or defending territory. Parrots are a type of bird, but they’re unique in their ability to imitate sounds. Maybe it\'s related to their social behavior?\n\nI remember reading somewhere that parrots are highly social animals. They live in flocks in the wild, so communication must be important for their survival. If they can mimic sounds, perhaps it helps them interact with each other. But how does that translate to talking with humans? Maybe they pick up on human speech because they\'re around humans a lot, especially in captivity. But why would they want to mimic humans?\n\nAnother thought: some animals learn by imitating others. For example, young parrots might learn to mimic thei

In [4]:
from langchain.tools import tool


@tool
def get_weather(location:str)-> str:
    """Get the weather at the location"""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])    

In [6]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    #View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'wstf9esyc', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.142002308, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006717228, 'prompt_tokens_details': None, 'queue_time': 0.048267382, 'total_time': 0.148719536}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e9ddf-09e1-7ac2-b

### Tool Execution Loop

In [7]:
#Step 1: Model generates tool calls
messages = [{"role":"user", "content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass results back to model for the final response
final_response  = model_with_tools.invoke(messages)
print(final_response.text)
#"The current weather in boston is 73˚F and sunny."

The weather in Boston is sunny. It's a great day to enjoy outdoor activities! ☀️


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': '7bww4t2d5', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 153, 'total_tokens': 257, 'completion_time': 0.172961924, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.006098499, 'prompt_tokens_details': None, 'queue_time': 0.281175759, 'total_time': 0.179060423}, 'model_name': 'qwen/qwen3-32b', 'syst